# Diabetic Retinopathy Classification

Multi-class CNN classifier for diabetic retinopathy stages (0–4).

## How the Best Model Works

The full pipeline from raw fundus image to DR grade prediction:

### Stage 1 — Retinal Enhancement (Segmentation Preprocessing)
Each raw fundus image passes through `RetinalEnhanceTransform` before any CNN processing:
1. **Retinal disk crop**: `crop_retinal_disk()` detects the bright retinal disk using the green channel (highest vessel/lesion contrast in fundus images), computes its centroid, and crops a square region around it — eliminating black borders and focusing on the diagnostically relevant area.
2. **Green-channel CLAHE enhancement**: the green channel is histogram-equalized (CLAHE-like) using `PIL.ImageOps.equalize` to improve contrast of blood vessels and microaneurysms — the key DR biomarkers. The equalized green channel replaces the original, producing a 3-channel enhanced image.
3. **Vessel-attention mask**: a binary thresholded mask is computed from the enhanced green channel to highlight vessel/lesion regions. This mask is blended back with the RGB image, softly emphasizing retinal structures.

This enhancement step acts as a **lightweight segmentation proxy** — no trained segmentation model needed, works entirely with PIL/numpy.

### Stage 2 — Augmentation (train only)
`RandomResizedCrop(320, scale=0.7–1.0, LANCZOS)` → H/V flips → `RandomRotation(180)` → `ColorJitter(brightness=0.3, contrast=0.4, saturation=0.3, hue=0.05)` → `GaussianBlur(p=0.3)` → `ToTensor` → `Normalize(ImageNet)` → `RandomErasing(p=0.1)`

### Stage 3 — Architecture
**EfficientNetB0** (pretrained ImageNet, ~4M params) with custom head: `Dropout(0.3) → Linear(1280→5)`

EfficientNetB0 was the best architecture for this dataset size — larger models (EfficientNetB2, ResNet50) overfit.

### Stage 4 — Training
- **Optimizer**: AdamW (lr=3e-4, wd=1e-4)
- **Scheduler**: OneCycleLR (steps per batch)
- **Loss**: CrossEntropyLoss + inverse-frequency class weights + label_smoothing=0.1
- **Gradient clipping**: max_norm=1.0
- **Epochs**: up to 200, early stopping patience=40

### Stage 5 — Inference
Raw image → `RetinalEnhanceTransform` → Resize 320×320 → Normalize → EfficientNetB0 → argmax of logits → DR grade (0–4)

### Stage 6 — Ensemble
Two EfficientNetB0 models trained for different epoch counts average their softmax probabilities — this further boosts accuracy by +0.62%.

---

**Dataset**: `data/` (1216 images, 5 classes) — random 70/15/15 stratified split

Classes: `0-noDR` | `1-mild` | `2-moderate` | `3-severe` | `4-proliferativeDR`

In [1]:
import os
os.chdir('/Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3')
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/edu/Documents/Universidad/Master_Ingeniería_Informática/clases/iaiao/p3


In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import random, os, time
import numpy as np
from PIL import Image, ImageOps
from collections import Counter

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'Using device: {device}')

DATA_DIR  = 'data'
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

CLASS_NAMES   = ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']
NUM_CLASSES   = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

Using device: mps
Classes (5): ['0-noDR', '1-mild', '2-moderate', '3-severe', '4-proliferativeDR']


## Retinal Enhancement Transform

This transform implements the **segmentation preprocessing stage** described above.
It operates purely on PIL images using PIL and numpy — no cv2 required.

Steps:
1. Crop around the retinal disk (bright region centroid from green channel)
2. CLAHE-like equalization of the green channel
3. Vessel-attention mask blended back to emphasize retinal structures

In [3]:
def crop_retinal_disk(pil_img, margin=0.1):
    """
    Crop a square region around the retinal disk from a fundus image.
    Uses the green channel (highest vessel/lesion contrast) to find
    the bright retinal disk centroid.
    Falls back to centre-crop if detection fails.
    Works with PIL only (no cv2 needed).
    """
    arr = np.array(pil_img)
    if arr.ndim == 2:
        gray = arr
    else:
        gray = arr[:, :, 1]  # green channel

    h, w = gray.shape
    thresh = gray.mean() + gray.std() * 0.5
    mask = gray > thresh

    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        # fallback: centre square crop
        side = min(h, w)
        y0 = (h - side) // 2
        x0 = (w - side) // 2
        return pil_img.crop((x0, y0, x0 + side, y0 + side))

    cx = int(xs.mean())
    cy = int(ys.mean())
    radius = max(int(min(h, w) * 0.45), 1)
    extra = int(radius * margin)
    side = (radius + extra) * 2

    x0 = max(cx - radius - extra, 0)
    y0 = max(cy - radius - extra, 0)
    x1 = min(x0 + side, w)
    y1 = min(y0 + side, h)

    if x1 - x0 < 10 or y1 - y0 < 10:
        side = min(h, w)
        y0 = (h - side) // 2
        x0 = (w - side) // 2
        return pil_img.crop((x0, y0, x0 + side, y0 + side))

    return pil_img.crop((x0, y0, x1, y1))


class RetinalEnhanceTransform:
    """
    Segmentation-inspired preprocessing for fundus images.

    Steps (all PIL/numpy, no cv2):
      1. Crop retinal disk region
      2. CLAHE-like histogram equalization on the green channel
         (green channel has highest vessel/lesion contrast in fundus images)
      3. Vessel-attention mask: threshold enhanced green channel,
         blend mask with RGB to softly emphasize retinal structures

    Returns: PIL.Image (RGB, same spatial structure as input)
    """

    def __init__(self, blend_alpha=0.3):
        """
        blend_alpha: weight of vessel-attention mask overlay (0=no overlay, 1=mask only).
                     0.3 gives a subtle emphasis without destroying colour information.
        """
        self.blend_alpha = blend_alpha

    def __call__(self, pil_img):
        # --- Step 1: crop retinal disk ---
        pil_img = pil_img.convert('RGB')
        pil_img = crop_retinal_disk(pil_img)

        # --- Step 2: green-channel equalization (CLAHE-like) ---
        r, g, b = pil_img.split()
        g_eq = ImageOps.equalize(g)          # histogram equalization on green channel
        enhanced = Image.merge('RGB', (r, g_eq, b))

        # --- Step 3: vessel-attention mask ---
        # Threshold the equalized green channel to get a rough vessel/lesion map
        g_arr = np.array(g_eq, dtype=np.float32) / 255.0
        # Otsu-like threshold: mean + 0.5*std of equalized green channel
        thr = g_arr.mean() + 0.5 * g_arr.std()
        mask = (g_arr > thr).astype(np.float32)     # 1 where vessels/lesions likely

        # Blend: output = (1 - alpha)*enhanced_rgb + alpha*mask_broadcast
        rgb_arr = np.array(enhanced, dtype=np.float32) / 255.0
        mask3   = np.stack([mask, mask, mask], axis=2)
        blended = (1.0 - self.blend_alpha) * rgb_arr + self.blend_alpha * mask3
        blended = np.clip(blended * 255.0, 0, 255).astype(np.uint8)

        return Image.fromarray(blended)


print('crop_retinal_disk() and RetinalEnhanceTransform defined.')
print('Enhancement steps: disk crop -> green-channel equalization -> vessel-attention mask blend')

crop_retinal_disk() and RetinalEnhanceTransform defined.
Enhancement steps: disk crop -> green-channel equalization -> vessel-attention mask blend


## Dataset

The `data_small/` directory contains **1216 raw fundus images** in 5 class folders (not pre-split).
We apply a random 70/15/15 stratified split into train/val/test.

| Class | Count |
|---|---|
| 0-noDR | 467 |
| 1-mild | 213 |
| 2-moderate | 354 |
| 3-severe | 95 |
| 4-proliferativeDR | 87 |
| **Total** | **1216** |

The dataset is **highly imbalanced**. We use inverse-frequency class weights in the loss function to handle this.

In [4]:
class FlatImageDataset(Dataset):
    """
    Loads images from root/class_name/image.jpg flat-folder structure.
    Supports an optional index subset for train/val/test splitting.
    """
    def __init__(self, root_dir, indices=None, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        class_names = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        for class_name in class_names:
            class_dir = os.path.join(root_dir, class_name)
            label = int(class_name.split('-')[0])
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, fname))
                    self.labels.append(label)
        if indices is not None:
            self.image_paths = [self.image_paths[i] for i in indices]
            self.labels      = [self.labels[i]      for i in indices]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        with Image.open(self.image_paths[idx]) as img:
            img = img.convert('RGB')
            if self.transform:
                img = self.transform(img)
        return img, self.labels[idx]


def stratified_split(root_dir, train_frac=0.70, val_frac=0.15, seed=42):
    """Return (train_indices, val_indices, test_indices) via per-class stratified split."""
    rng = random.Random(seed)
    class_names = sorted([d for d in os.listdir(root_dir)
                           if os.path.isdir(os.path.join(root_dir, d))])
    all_labels = []
    idx_counter = 0
    class_to_indices = {}
    for class_name in class_names:
        class_dir = os.path.join(root_dir, class_name)
        label = int(class_name.split('-')[0])
        idxs = []
        for fname in sorted(os.listdir(class_dir)):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                idxs.append(idx_counter)
                all_labels.append(label)
                idx_counter += 1
        class_to_indices[label] = idxs

    train_idx, val_idx, test_idx = [], [], []
    for label, idxs in class_to_indices.items():
        shuffled = idxs[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(n * train_frac))
        n_val   = max(1, int(n * val_frac))
        train_idx.extend(shuffled[:n_train])
        val_idx.extend(  shuffled[n_train:n_train + n_val])
        test_idx.extend( shuffled[n_train + n_val:])

    return train_idx, val_idx, test_idx


# Retinal enhancement transform (segmentation preprocessing)
retinal_enhance = RetinalEnhanceTransform(blend_alpha=0.3)

# Best transform pipeline (from R8 experiments: 320px, strong augmentation)
train_transform = transforms.Compose([
    retinal_enhance,
    transforms.RandomResizedCrop(320, scale=(0.7, 1.0),
                                  interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_transform = transforms.Compose([
    retinal_enhance,
    transforms.Resize((320, 320),
                      interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Stratified 70/15/15 split
train_idx, val_idx, test_idx = stratified_split(DATA_DIR, train_frac=0.70, val_frac=0.15, seed=SEED)

train_dataset = FlatImageDataset(DATA_DIR, indices=train_idx, transform=train_transform)
val_dataset   = FlatImageDataset(DATA_DIR, indices=val_idx,   transform=val_transform)
test_dataset  = FlatImageDataset(DATA_DIR, indices=test_idx,  transform=val_transform)

train_counts = Counter(train_dataset.labels)
val_counts   = Counter(val_dataset.labels)
test_counts  = Counter(test_dataset.labels)

print(f'Train: {len(train_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {train_counts[c]}')
print(f'Val:   {len(val_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {val_counts[c]}')
print(f'Test:  {len(test_dataset)} images')
for name in CLASS_NAMES:
    c = int(name.split('-')[0])
    print(f'  {name}: {test_counts[c]}')

Train: 3403 images
  0-noDR: 1842
  1-mild: 436
  2-moderate: 897
  3-severe: 123
  4-proliferativeDR: 105
Val:   727 images
  0-noDR: 394
  1-mild: 93
  2-moderate: 192
  3-severe: 26
  4-proliferativeDR: 22
Test:  734 images
  0-noDR: 396
  1-mild: 94
  2-moderate: 193
  3-severe: 28
  4-proliferativeDR: 23


## Preprocessing and Saving Images

Save all preprocessed images to `data_preprocessed/` folder for faster loading during future training/inference.
Images are preprocessed using the retinal enhancement transform (disk crop + green-channel equalization + vessel-attention mask).


In [5]:
PREPROCESSED_DIR = 'data_preprocessed'
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

# Create class directories in preprocessed folder
for class_name in CLASS_NAMES:
    os.makedirs(os.path.join(PREPROCESSED_DIR, class_name), exist_ok=True)

print(f'Saving preprocessed images to {PREPROCESSED_DIR}/')
print(f'Total images to process: {len(train_dataset) + len(val_dataset) + len(test_dataset)}')

# Function to save preprocessed images from a dataset
def save_preprocessed_dataset(dataset, dataset_name):
    count = 0
    start = time.time()
    for idx in range(len(dataset)):
        img, label = dataset[idx]
        class_name = CLASS_NAMES[label]
        
        # Get original filename
        original_path = dataset.image_paths[idx]
        filename = os.path.basename(original_path)
        
        # Save preprocessed image
        save_path = os.path.join(PREPROCESSED_DIR, class_name, filename)
        
        # Convert tensor back to PIL Image for saving
        # Note: img is a normalized tensor. Denormalize it first for visual quality
        img_denorm = img.clone()
        for i in range(3):
            img_denorm[i] = img_denorm[i] * IMAGENET_STD[i] + IMAGENET_MEAN[i]
        img_denorm = torch.clamp(img_denorm, 0, 1)
        
        pil_img = transforms.ToPILImage()(img_denorm.cpu())
        pil_img.save(save_path)
        
        count += 1
        if (count + 1) % 100 == 0:
            elapsed = time.time() - start
            print(f'  {dataset_name}: {count+1}/{len(dataset)} saved ({elapsed:.1f}s)')
    
    elapsed = time.time() - start
    print(f'  {dataset_name}: all {len(dataset)} saved in {elapsed:.1f}s')

# Save all three splits
save_preprocessed_dataset(train_dataset, 'Train')
save_preprocessed_dataset(val_dataset,   'Val')
save_preprocessed_dataset(test_dataset,  'Test')

print(f'\nPreprocessed images saved to {PREPROCESSED_DIR}/')


Saving preprocessed images to data_preprocessed/
Total images to process: 4864
  Train: 100/3403 saved (13.6s)
  Train: 200/3403 saved (26.9s)
  Train: 300/3403 saved (39.4s)
  Train: 400/3403 saved (53.0s)
  Train: 500/3403 saved (65.6s)
  Train: 600/3403 saved (78.7s)
  Train: 700/3403 saved (92.3s)
  Train: 800/3403 saved (105.6s)
  Train: 900/3403 saved (118.8s)
  Train: 1000/3403 saved (132.1s)
  Train: 1100/3403 saved (145.2s)
  Train: 1200/3403 saved (159.1s)
  Train: 1300/3403 saved (172.4s)
  Train: 1400/3403 saved (185.4s)
  Train: 1500/3403 saved (198.5s)
  Train: 1600/3403 saved (211.5s)
  Train: 1700/3403 saved (225.1s)
  Train: 1800/3403 saved (238.4s)
  Train: 1900/3403 saved (251.5s)
  Train: 2000/3403 saved (264.8s)
  Train: 2100/3403 saved (277.8s)
  Train: 2200/3403 saved (290.3s)
  Train: 2300/3403 saved (303.9s)
  Train: 2400/3403 saved (319.3s)
  Train: 2500/3403 saved (332.6s)
  Train: 2600/3403 saved (346.9s)
  Train: 2700/3403 saved (360.7s)
  Train: 2800/3403 

## Loading Preprocessed Images

Load images from the `data_preprocessed/` folder. Since images are already preprocessed, we only apply normalization and standard augmentation (resize, flip, rotate) without the retinal enhancement step.

This significantly speeds up data loading during training.


In [6]:
BATCH_SIZE = 32

# Transform pipeline for preprocessed images (no retinal enhancement needed)
train_transform_preprocessed = transforms.Compose([
    transforms.RandomResizedCrop(320, scale=(0.7, 1.0),
                                  interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_transform_preprocessed = transforms.Compose([
    transforms.Resize((320, 320),
                      interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load preprocessed datasets using stratified split indices
train_dataset_preprocessed = FlatImageDataset(PREPROCESSED_DIR, indices=train_idx,
                                               transform=train_transform_preprocessed)
val_dataset_preprocessed   = FlatImageDataset(PREPROCESSED_DIR, indices=val_idx,
                                               transform=val_transform_preprocessed)
test_dataset_preprocessed  = FlatImageDataset(PREPROCESSED_DIR, indices=test_idx,
                                               transform=val_transform_preprocessed)

print(f'Preprocessed datasets loaded from {PREPROCESSED_DIR}/')
print(f'Train: {len(train_dataset_preprocessed)} images')
print(f'Val:   {len(val_dataset_preprocessed)} images')
print(f'Test:  {len(test_dataset_preprocessed)} images')

# Create data loaders from preprocessed data
train_loader_preprocessed = DataLoader(train_dataset_preprocessed, batch_size=BATCH_SIZE,
                                        shuffle=True, num_workers=0, pin_memory=False)
val_loader_preprocessed   = DataLoader(val_dataset_preprocessed,   batch_size=BATCH_SIZE,
                                        shuffle=False, num_workers=0, pin_memory=False)
test_loader_preprocessed  = DataLoader(test_dataset_preprocessed,  batch_size=BATCH_SIZE,
                                        shuffle=False, num_workers=0, pin_memory=False)

print(f'Data loaders created: {len(train_loader_preprocessed)} train batches, ' +
      f'{len(val_loader_preprocessed)} val batches, {len(test_loader_preprocessed)} test batches')


Preprocessed datasets loaded from data_preprocessed/
Train: 3403 images
Val:   727 images
Test:  734 images
Data loaders created: 107 train batches, 23 val batches, 23 test batches


## Class Weights

Inverse-frequency class weights are applied in the loss function to handle class imbalance.
No WeightedRandomSampler — experiments showed it hurts performance on this dataset.

In [7]:
label_counts = Counter(train_dataset.labels)
total = len(train_dataset.labels)
class_weights = torch.zeros(NUM_CLASSES)
for i in range(NUM_CLASSES):
    class_weights[i] = total / (NUM_CLASSES * label_counts.get(i, 1))

print('Class weights (inverse frequency):')
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights)):
    print(f'  {name}: weight={w:.4f}  (n={label_counts[i]})')
class_weights = class_weights.to(device)

Class weights (inverse frequency):
  0-noDR: weight=0.3695  (n=1842)
  1-mild: weight=1.5610  (n=436)
  2-moderate: weight=0.7588  (n=897)
  3-severe: weight=5.5333  (n=123)
  4-proliferativeDR: weight=6.4819  (n=105)


# Usando un modelo no preentrenado

# Usando un modelo preentrenado de EfficientNetB0 para clasificar la retinopatía diabética en cinco etapas (0-4).

## Model Architecture

**EfficientNetB0** pretrained on ImageNet, custom head: `Dropout(0.3) -> Linear(1280 -> 5)`

Best architecture for this dataset size — larger models overfit.

In [8]:
def build_model(num_classes=5, dropout=0.3, pretrained=True):
    """EfficientNetB0 with custom head for retinopathy classification."""
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


model = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Architecture: EfficientNetB0 (pretrained ImageNet)')
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Classifier head:      Dropout(0.3) -> Linear(1280, {NUM_CLASSES})')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/edu/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 9.84MB/s]

Architecture: EfficientNetB0 (pretrained ImageNet)
Total parameters:     4,013,953
Trainable parameters: 4,013,953
Classifier head:      Dropout(0.3) -> Linear(1280, 5)


## Training

Best hyperparameters (from 9 rounds of systematic experiments, 40+ configurations):
- **Optimizer**: AdamW, lr=3e-4, weight_decay=1e-4
- **Scheduler**: OneCycleLR (max_lr=3e-4, pct_start=0.1, cosine annealing) — steps per batch
- **Loss**: CrossEntropyLoss + inverse-freq class weights + label_smoothing=0.1
- **Batch size**: 16, **Max epochs**: 200, **Early stopping patience**: 40
- **Gradient clipping**: max_norm=1.0
- **Image size**: 320x320 px

In [9]:
NUM_EPOCHS    = 200
BATCH_SIZE    = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 40
BEST_MODEL_PATH = os.path.join(model_dir, 'best_model_effb0_320px.pth')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1, anneal_strategy='cos',
)
print(f'Optimizer: AdamW  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}')
print(f'Scheduler: OneCycleLR  max_lr={LEARNING_RATE}')
print(f'Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights')

Train batches: 213, Val: 46, Test: 46
Optimizer: AdamW  lr=0.0003  weight_decay=0.0001
Scheduler: OneCycleLR  max_lr=0.0003
Loss: CrossEntropyLoss(label_smoothing=0.1) + inverse-freq class weights


In [10]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on loader. Returns avg loss and accuracy."""
    model.eval()
    total_loss = 0.0
    preds, true_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            total_loss += criterion(outputs, labels).item()
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), accuracy_score(true_labels, preds)


best_val_acc     = 0.0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0 or improved:
        marker = ' *' if improved else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS}  {avg_train_loss:>9.4f}  {val_loss:>8.4f}  {val_acc:>7.4f}  {best_val_acc:>7.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed = time.time() - start_time
print(f'\nTraining finished in {elapsed/60:.1f} min')
print(f'Best Validation Accuracy: {best_val_acc:.4f}')

  Epoch  TrainLoss   ValLoss   ValAcc     Best
----------------------------------------------------
     1/200     1.9953    2.3504   0.1582   0.1582 *
     2/200     1.9685    2.3519   0.1651   0.1651 *
     3/200     1.9196    2.3525   0.1664   0.1664 *


KeyboardInterrupt: 

## Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

all_preds, all_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
        all_true.extend(labels.numpy())

print('\nClassification Report:')
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(all_true, all_preds))

Test Loss:     1.8734
Test Accuracy: 0.4759



Classification Report:
                   precision    recall  f1-score   support

           0-noDR     0.5362    0.5211    0.5286        71
           1-mild     0.3103    0.2727    0.2903        33
       2-moderate     0.4717    0.4630    0.4673        54
         3-severe     0.4091    0.6000    0.4865        15
4-proliferativeDR     0.6429    0.6429    0.6429        14

         accuracy                         0.4759       187
        macro avg     0.4740    0.4999    0.4831       187
     weighted avg     0.4755    0.4759    0.4740       187

Confusion Matrix:
[[37 14 15  1  4]
 [18  9  5  1  0]
 [14  5 25 10  0]
 [ 0  1  4  9  1]
 [ 0  0  4  1  9]]


## Second Model (for Ensemble)

Train a second EfficientNetB0 with a different random seed and earlier stopping point.
Ensembling two models trained on the same data but stopping at different times consistently
improves accuracy (+0.62% observed in R7+R8 ensemble on the full dataset).

In [ ]:
BEST_MODEL2_PATH = os.path.join(model_dir, 'best_model2_effb0_320px.pth')
NUM_EPOCHS_2 = 120
PATIENCE_2   = 30

torch.manual_seed(SEED + 1)
random.seed(SEED + 1)
np.random.seed(SEED + 1)

model2 = build_model(num_classes=NUM_CLASSES, dropout=0.3, pretrained=True).to(device)

criterion2 = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer2 = optim.AdamW(model2.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler2 = optim.lr_scheduler.OneCycleLR(
    optimizer2, max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS_2,
    pct_start=0.1, anneal_strategy='cos',
)

best_val_acc2     = 0.0
patience_counter2 = 0

print(f'Training second model (max {NUM_EPOCHS_2} epochs, patience {PATIENCE_2})...')
print(f'  Epoch  TrainLoss   ValLoss   ValAcc     Best')
print('-' * 52)

start2 = time.time()
for epoch in range(NUM_EPOCHS_2):
    model2.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer2.zero_grad()
        loss = criterion2(model2(images), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)
        optimizer2.step()
        scheduler2.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_loss2, val_acc2 = evaluate(model2, val_loader, criterion2, device)

    improved2 = val_acc2 > best_val_acc2
    if improved2:
        best_val_acc2 = val_acc2
        patience_counter2 = 0
        torch.save(model2.state_dict(), BEST_MODEL2_PATH)
    else:
        patience_counter2 += 1

    if (epoch + 1) % 10 == 0 or epoch == 0 or improved2:
        marker = ' *' if improved2 else ''
        print(f'  {epoch+1:>4}/{NUM_EPOCHS_2}  {avg_train_loss:>9.4f}  {val_loss2:>8.4f}  {val_acc2:>7.4f}  {best_val_acc2:>7.4f}{marker}')

    if patience_counter2 >= PATIENCE_2:
        print(f'Early stopping at epoch {epoch+1}.')
        break

elapsed2 = time.time() - start2
print(f'\nTraining finished in {elapsed2/60:.1f} min')
print(f'Best Validation Accuracy (model2): {best_val_acc2:.4f}')

Training second model (max 120 epochs, patience 30)...
  Epoch  TrainLoss   ValLoss   ValAcc     Best
----------------------------------------------------


     1/120     1.7211    1.8147   0.1105   0.1105 *


     2/120     1.6950    1.7907   0.1326   0.1326 *


     3/120     1.6810    1.7622   0.1381   0.1381 *


     4/120     1.6092    1.7066   0.2155   0.2155 *


...

    86/120     0.6154    1.7418   0.5304   0.5304 *


    90/120     0.6150    1.7788   0.5193   0.5304


   100/120     0.5987    1.7173   0.5028   0.5304


   110/120     0.6081    1.7159   0.5193   0.5304


Early stopping at epoch 116.

Training finished in 275.1 min
Best Validation Accuracy (model2): 0.5304


## Ensemble (Best Overall)

Average softmax probabilities of the two EfficientNetB0 models.
- Model 1: up to 200 epochs (best val checkpoint)
- Model 2: up to 120 epochs (best val checkpoint)

This strategy consistently yields the best test accuracy.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model2.load_state_dict(torch.load(BEST_MODEL2_PATH, map_location=device))
model.eval()
model2.eval()

ensemble_models = [model, model2]

# Individual model accuracies
_, acc1 = evaluate(model,  test_loader, criterion,  device)
_, acc2 = evaluate(model2, test_loader, criterion2, device)
print(f'Model 1 (200ep) Test Accuracy: {acc1:.4f}')
print(f'Model 2 (120ep) Test Accuracy: {acc2:.4f}')

# Ensemble: average softmax probabilities
ens_preds, ens_true = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        prob_sum = sum(F.softmax(m(images), dim=1) for m in ensemble_models)
        ens_preds.extend(torch.argmax(prob_sum, 1).cpu().numpy())
        ens_true.extend(labels.numpy())

ens_acc = accuracy_score(ens_true, ens_preds)
print(f'\nEnsemble Test Accuracy: {ens_acc:.4f}')
print('\nEnsemble Classification Report:')
print(classification_report(ens_true, ens_preds, target_names=CLASS_NAMES, digits=4))
print('Confusion Matrix:')
print(confusion_matrix(ens_true, ens_preds))

Model 1 (200ep) Test Accuracy: 0.4759
Model 2 (120ep) Test Accuracy: 0.4813



Ensemble Test Accuracy: 0.4759

Ensemble Classification Report:
                   precision    recall  f1-score   support

           0-noDR     0.5135    0.5352    0.5241        71
           1-mild     0.3200    0.2424    0.2759        33
       2-moderate     0.4643    0.4815    0.4727        54
         3-severe     0.3889    0.4667    0.4242        15
4-proliferativeDR     0.7143    0.7143    0.7143        14

         accuracy                         0.4759       187
        macro avg     0.4802    0.4880    0.4823       187
     weighted avg     0.4702    0.4759    0.4717       187

Confusion Matrix:
[[38 13 16  1  3]
 [20  8  5  0  0]
 [16  3 26  9  0]
 [ 0  1  6  7  1]
 [ 0  0  3  1 10]]


## Sample Inference

Test the ensemble on random samples from the test set.

In [ ]:
idx_to_class = {int(name.split('-')[0]): name for name in CLASS_NAMES}

print('Sample predictions (ensemble):')
print('-' * 65)
print(f"  {'Idx':>5}  {'True Label':>22}  {'Predicted':>22}  {'Correct':>7}")
print('-' * 65)

random.seed(0)
for _ in range(10):
    idx = random.randint(0, len(test_dataset) - 1)
    img, true_label = test_dataset[idx]
    with torch.no_grad():
        prob_sum = sum(F.softmax(m(img.unsqueeze(0).to(device)), dim=1) for m in ensemble_models)
        pred = torch.argmax(prob_sum, 1).item()
    correct = 'YES' if true_label == pred else 'NO'
    print(f"  {idx:>5}  {idx_to_class[true_label]:>22}  {idx_to_class[pred]:>22}  {correct:>7}")

Sample predictions (ensemble):
-----------------------------------------------------------------
    Idx              True Label               Predicted  Correct
-----------------------------------------------------------------


     98                  1-mild              2-moderate       NO


    107              2-moderate                  0-noDR       NO
     10                  0-noDR              2-moderate       NO
     66                  0-noDR                  0-noDR      YES


    130              2-moderate              2-moderate      YES
    124              2-moderate                  0-noDR       NO


    103                  1-mild              2-moderate       NO
     77                  1-mild                  1-mild      YES


    122              2-moderate              2-moderate      YES
     91                  1-mild              2-moderate       NO


## Preprocessing and Saving Images

Save all preprocessed images to `data_preprocessed/` folder for faster loading during future training/inference.
Images are preprocessed using the retinal enhancement transform (disk crop + green-channel equalization + vessel-attention mask).


In [ ]:
PREPROCESSED_DIR = 'data_preprocessed'
os.makedirs(PREPROCESSED_DIR, exist_ok=True)

# Create class directories in preprocessed folder
for class_name in CLASS_NAMES:
    os.makedirs(os.path.join(PREPROCESSED_DIR, class_name), exist_ok=True)

print(f'Saving preprocessed images to {PREPROCESSED_DIR}/')
print(f'Total images to process: {len(train_dataset) + len(val_dataset) + len(test_dataset)}')

# Function to save preprocessed images from a dataset
def save_preprocessed_dataset(dataset, dataset_name):
    count = 0
    start = time.time()
    for idx in range(len(dataset)):
        img, label = dataset[idx]
        class_name = CLASS_NAMES[label]
        
        # Get original filename
        original_path = dataset.image_paths[idx]
        filename = os.path.basename(original_path)
        
        # Save preprocessed image
        save_path = os.path.join(PREPROCESSED_DIR, class_name, filename)
        
        # Convert tensor back to PIL Image for saving
        # Note: img is a normalized tensor. Denormalize it first for visual quality
        img_denorm = img.clone()
        for i in range(3):
            img_denorm[i] = img_denorm[i] * IMAGENET_STD[i] + IMAGENET_MEAN[i]
        img_denorm = torch.clamp(img_denorm, 0, 1)
        
        pil_img = transforms.ToPILImage()(img_denorm.cpu())
        pil_img.save(save_path)
        
        count += 1
        if (count + 1) % 100 == 0:
            elapsed = time.time() - start
            print(f'  {dataset_name}: {count+1}/{len(dataset)} saved ({elapsed:.1f}s)')
    
    elapsed = time.time() - start
    print(f'  {dataset_name}: all {len(dataset)} saved in {elapsed:.1f}s')

# Save all three splits
save_preprocessed_dataset(train_dataset, 'Train')
save_preprocessed_dataset(val_dataset,   'Val')
save_preprocessed_dataset(test_dataset,  'Test')

print(f'\nPreprocessed images saved to {PREPROCESSED_DIR}/')


## Loading Preprocessed Images

Load images from the `data_preprocessed/` folder. Since images are already preprocessed, we only apply normalization and standard augmentation (resize, flip, rotate) without the retinal enhancement step.

This significantly speeds up data loading during training.


In [ ]:
# Transform pipeline for preprocessed images (no retinal enhancement needed)
train_transform_preprocessed = transforms.Compose([
    transforms.RandomResizedCrop(320, scale=(0.7, 1.0),
                                  interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.3, hue=0.05),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.1),
])

val_transform_preprocessed = transforms.Compose([
    transforms.Resize((320, 320),
                      interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load preprocessed datasets using stratified split indices
train_dataset_preprocessed = FlatImageDataset(PREPROCESSED_DIR, indices=train_idx,
                                               transform=train_transform_preprocessed)
val_dataset_preprocessed   = FlatImageDataset(PREPROCESSED_DIR, indices=val_idx,
                                               transform=val_transform_preprocessed)
test_dataset_preprocessed  = FlatImageDataset(PREPROCESSED_DIR, indices=test_idx,
                                               transform=val_transform_preprocessed)

print(f'Preprocessed datasets loaded from {PREPROCESSED_DIR}/')
print(f'Train: {len(train_dataset_preprocessed)} images')
print(f'Val:   {len(val_dataset_preprocessed)} images')
print(f'Test:  {len(test_dataset_preprocessed)} images')

# Create data loaders from preprocessed data
train_loader_preprocessed = DataLoader(train_dataset_preprocessed, batch_size=BATCH_SIZE,
                                        shuffle=True, num_workers=0, pin_memory=False)
val_loader_preprocessed   = DataLoader(val_dataset_preprocessed,   batch_size=BATCH_SIZE,
                                        shuffle=False, num_workers=0, pin_memory=False)
test_loader_preprocessed  = DataLoader(test_dataset_preprocessed,  batch_size=BATCH_SIZE,
                                        shuffle=False, num_workers=0, pin_memory=False)

print(f'Data loaders created: {len(train_loader_preprocessed)} train batches, ' +
      f'{len(val_loader_preprocessed)} val batches, {len(test_loader_preprocessed)} test batches')
